# Notebook 3A — Adoption Dating & Panel Construction
### Study: *The Second-Order Compute Footprint of AI-Coding-Tool Adoption in Open Source*

G1 **passed** (precision 1.0, 95% CI [0.975, 1.0]; recall 1.0, CI [0.975, 1.0]; κ = 0.991). The detector's high-confidence signals are validated, so the treated group = **repos with high-confidence adoption signals** (340 repos; 339 with retrievable CI).

This notebook builds the **repo × month panels** that the difference-in-differences analysis (Notebook 3B) runs on. It is the heavy, resumable collection step. Split from the analysis (3B) so a disconnect never costs analysis work — the same resilience principle used throughout.

**What it does**
1. **Assign adoption dates.** Each treated repo's adoption date = the earliest high-confidence signal (earliest AI co-author commit, or the commit that introduced the AI-tool config artifact).
2. **Define cohorts & windows** per the locked design:
   - *Activity outcomes* (commit frequency, code churn, PR volume) → all 340 treated, **full window**, from git/PR history (does not age out).
   - *CI-minutes outcome* → treated with **≥ 9 months** retrievable CI history (~223), **scoped window**, from Actions run durations.
3. **Assign control pseudo-adoption dates** so controls align to the DiD event time.
4. **Collect & build the panels** (resumable, checkpointed to Drive), writing tidy `repo × month` tables.

**Standards:** ACM SIGSOFT *Repository Mining*, *Data Science*, *Longitudinal*. Construct/threats handling per the pre-registration. The CI-minutes outcome carries the documented ~14-month Actions-retention boundary (deviation already logged in NB2).

> **Honest scoping carried from NB2:** CI-minutes → energy → CO2e uses an estimation proxy (Green Algorithms / SCI ISO/IEC 21031:2024 / Cloud Carbon Footprint). That conversion lives in the carbon notebook; 3A only collects the *minutes*. Recall is "recall w.r.t. human-detectable adoption"; treated = signal-validated adopters.


---
## Cell 0 — Re-attach environment (Drive, config, cached client)

Re-establishes everything from NB1/NB2. Run NB1 and NB2 at least once first.


In [ ]:

%pip install -q requests requests-cache tenacity pandas pyarrow tqdm python-dateutil

import os, json, time, math, pathlib, datetime, random
import requests, requests_cache
import pandas as pd
from dateutil import parser as dtp
from tenacity import retry, wait_exponential, stop_after_attempt, retry_if_exception_type

from google.colab import drive, userdata
drive.mount('/content/drive')

PROJECT_ROOT = pathlib.Path('/content/drive/MyDrive/ai_coding_carbon')
DIRS = {k: PROJECT_ROOT / k for k in
        ['raw','cache','panels','cohort','results','figures','logs','frame','gold']}
for d in [PROJECT_ROOT, *DIRS.values()]: d.mkdir(parents=True, exist_ok=True)

CONFIG = json.loads((PROJECT_ROOT / 'config.json').read_text())
random.seed(CONFIG['random_seed'])

GITHUB_TOKEN = (userdata.get('GITHUB_TOKEN') or '').strip()
assert GITHUB_TOKEN, 'No GITHUB_TOKEN in Colab Secrets.'
API = 'https://api.github.com'
HEADERS = {'Accept':'application/vnd.github+json','X-GitHub-Api-Version':'2022-11-28',
           'Authorization': f'Bearer {GITHUB_TOKEN}'}
_p = requests.get(f'{API}/user', headers=HEADERS, timeout=30)
assert _p.status_code == 200, f'Auth failed {_p.status_code}'
print('Authenticated as:', _p.json()['login'])

session = requests_cache.CachedSession(
    cache_name=str(DIRS['cache']/'github_http_cache'),
    backend='sqlite', expire_after=None, allowable_codes=(200,), stale_if_error=True)

class RateLimited(Exception): pass
@retry(retry=retry_if_exception_type(RateLimited),
       wait=wait_exponential(multiplier=2,min=4,max=120), stop=stop_after_attempt(8))
def gh_get(url, params=None):
    r = session.get(url, headers=HEADERS, params=params, timeout=60)
    if r.status_code in (403,429):
        rem = r.headers.get('X-RateLimit-Remaining')
        if rem=='0' or 'rate limit' in r.text.lower() or 'secondary' in r.text.lower():
            reset=int(r.headers.get('X-RateLimit-Reset',time.time()+60))
            sl=max(reset-time.time(),5)+2; print(f'  [rate limit] sleep {sl:.0f}s'); time.sleep(sl)
            raise RateLimited()
    r.raise_for_status(); return r
def gh_json(url, params=None): return gh_get(url, params=params).json()

def append_jsonl(path, rec):
    with open(path,'a') as f:
        f.write(json.dumps(rec)); f.write(chr(10))
def load_jsonl(path):
    recs=[]
    if not pathlib.Path(path).exists(): return recs
    for line in open(path):
        line=line.strip()
        if line:
            try: recs.append(json.loads(line))
            except json.JSONDecodeError: pass
    return recs
def load_ledger(p,d): return json.loads(pathlib.Path(p).read_text()) if pathlib.Path(p).exists() else d
def save_ledger(p,o): pathlib.Path(p).write_text(json.dumps(o,indent=2))

frame = pd.read_parquet(DIRS['frame']/'frame_final.parquet')
print('Frame loaded:', len(frame), 'repos |',
      int(frame['high_conf_signal'].sum()), 'treated')


---
## Cell 1 — Assign adoption dates to treated repos

The **adoption date** is the earliest high-confidence signal we can date:
- **commit trailers:** the date of the earliest commit whose message carries an AI co-author/trailer signal (already captured in `commit_trailer_hits`).
- **config artifacts:** the author date of the commit that first *introduced* the artifact path (queried via the commits API filtered by path, taking the oldest).

We take the **earliest** of the available dated signals as the adoption date. Repos whose only signal can't be dated are flagged and excluded from event-time analyses (documented).


In [ ]:

adopt_path = DIRS['cohort']/'adoption_dates.jsonl'
adopt_led  = DIRS['cohort']/'adoption_ledger.json'
led = load_ledger(adopt_led, {"done": []})
done = set(led["done"])

treated = frame[frame['high_conf_signal']].copy()

def earliest_trailer_date(hits):
    ds = []
    for h in (hits or []):
        try: ds.append(dtp.parse(h['date']))
        except Exception: pass
    return min(ds) if ds else None

def artifact_intro_date(full_name, artifact):
    """Date of the OLDEST commit touching the artifact path = when it was introduced."""
    try:
        commits = gh_json(f"{API}/repos/{full_name}/commits",
                          params={"path": artifact, "per_page": 100})
        if not commits: return None
        dates = [dtp.parse(c["commit"]["author"]["date"]) for c in commits]
        return min(dates) if dates else None
    except Exception:
        return None

todo = [r for r in treated.to_dict('records') if r['full_name'] not in done]
print(f"Dating {len(todo)} treated repos (done {len(done)})")

for i, r in enumerate(todo):
    fn = r['full_name']
    cands = []
    td = earliest_trailer_date(r.get('commit_trailer_hits'))
    if td: cands.append(('commit_trailer', td))
    for art in (r.get('config_artifacts') or []):
        ad = artifact_intro_date(fn, art)
        if ad: cands.append((f'artifact:{art}', ad))
    if cands:
        src, d = min(cands, key=lambda x: x[1])
        rec = {"full_name": fn, "adoption_date": d.isoformat(),
               "adoption_source": src, "n_dated_signals": len(cands)}
    else:
        rec = {"full_name": fn, "adoption_date": None,
               "adoption_source": None, "n_dated_signals": 0}
    append_jsonl(adopt_path, rec)
    done.add(fn); led["done"]=sorted(done)
    if i % 25 == 0: save_ledger(adopt_led, led); print(f"  [{i+1}/{len(todo)}] {fn} -> {rec['adoption_date']}")
save_ledger(adopt_led, led)

adf = pd.DataFrame(load_jsonl(adopt_path)).drop_duplicates('full_name')
dated = adf['adoption_date'].notna()
print(f"\nDated {int(dated.sum())}/{len(adf)} treated repos.")
print("Adoption-source breakdown:"); print(adf['adoption_source'].value_counts(dropna=False))
adf.to_parquet(DIRS['cohort']/'adoption_dates.parquet', index=False)


---
## Cell 2 — Define cohorts, windows, and control pseudo-dates

We build the analysis cohorts per the locked design and assign each **control** a pseudo-adoption date so it has a comparable event time. To avoid biasing the DiD, control pseudo-dates are drawn from the **treated adoption-date distribution** (sampling with replacement), so treated and control event-time placement match by construction.


In [ ]:

adf = pd.read_parquet(DIRS['cohort']/'adoption_dates.parquet')
frame = pd.read_parquet(DIRS['frame']/'frame_final.parquet')

treated = frame[frame['high_conf_signal']].merge(
    adf[['full_name','adoption_date','adoption_source']], on='full_name', how='left')
treated = treated[treated['adoption_date'].notna()].copy()
treated['adoption_date'] = pd.to_datetime(treated['adoption_date'], utc=True)
treated['is_treated'] = True

controls = frame[~frame['high_conf_signal']].copy()
controls['is_treated'] = False

# control pseudo-dates ~ treated adoption-date distribution
rng = random.Random(CONFIG['random_seed'])
treated_dates = list(treated['adoption_date'])
controls['adoption_date'] = [rng.choice(treated_dates) for _ in range(len(controls))]
controls['adoption_source'] = 'pseudo'

PRE = CONFIG['pre_period_days']; POST = CONFIG['post_period_days']

# CI-eligibility: >=9 months retrievable CI history (carbon outcome subset)
def months_back(a):
    if not isinstance(a, dict): return None
    ts=a.get('oldest')
    if not ts: return None
    return round((pd.Timestamp.now(tz='UTC')-pd.to_datetime(ts,utc=True)).days/30.44,1)
for dfx in (treated, controls):
    dfx['ci_oldest_months_back'] = dfx['actions_depth'].map(months_back)

cohort = pd.concat([treated, controls], ignore_index=True)
cohort['ci_eligible'] = cohort['ci_oldest_months_back'].fillna(0) >= 9
cohort.to_parquet(DIRS['cohort']/'cohort.parquet', index=False)

print("COHORT SUMMARY")
print(f"  treated (dated):     {int(cohort['is_treated'].sum())}")
print(f"  controls:            {int((~cohort['is_treated']).sum())}")
print(f"  CI-eligible treated:  {int((cohort['is_treated'] & cohort['ci_eligible']).sum())}")
print(f"  CI-eligible controls: {int((~cohort['is_treated'] & cohort['ci_eligible']).sum())}")
print(f"  activity window: +/- pre={PRE}d / post={POST}d around (pseudo-)adoption date")


---
## Cell 3 — Collect commit history → monthly activity panel (all cohort repos)

For every cohort repo we page the commits API across the study window and aggregate to **monthly commit counts** and **code churn** (additions+deletions, via per-commit stats sampled to control API cost — see note). This is the activity backbone for RQ1; it comes from git history and does **not** age out, so it uses the full window. Resumable per repo.

> **API-cost note:** full per-commit churn (additions/deletions) requires one call per commit, which is prohibitive at scale. We collect **commit counts** exhaustively (cheap, list endpoint) and collect **churn** via the `stats/commit_activity` and `stats/code_frequency` *weekly* endpoints (one call per repo), which GitHub computes server-side. This keeps the panel affordable and is documented as a measurement choice.


In [ ]:

cohort = pd.read_parquet(DIRS['cohort']/'cohort.parquet')
commits_panel_path = DIRS['panels']/'commits_monthly.jsonl'
churn_panel_path   = DIRS['panels']/'churn_weekly.jsonl'
act_led = DIRS['panels']/'activity_ledger.json'
led = load_ledger(act_led, {"commits_done": [], "churn_done": []})
cdone=set(led["commits_done"]); hdone=set(led["churn_done"])

W0 = pd.Timestamp(CONFIG['study_window'][0], tz='UTC')
W1 = pd.Timestamp(CONFIG['study_window'][1], tz='UTC')

def monthly_commit_counts(full_name):
    counts={}; page=1
    while page<=20:
        cs = gh_json(f"{API}/repos/{full_name}/commits",
                     params={"since":W0.isoformat(),"until":W1.isoformat(),
                             "per_page":100,"page":page})
        if not cs: break
        for c in cs:
            d=dtp.parse(c["commit"]["author"]["date"])
            key=f"{d.year}-{d.month:02d}"; counts[key]=counts.get(key,0)+1
        if len(cs)<100: break
        page+=1
    return counts

def weekly_code_frequency(full_name):
    # stats/code_frequency: list of [week_unix, additions, deletions(neg)]
    try:
        data = gh_json(f"{API}/repos/{full_name}/stats/code_frequency")
        if isinstance(data, list): return data
    except Exception: pass
    return []

repos = cohort['full_name'].tolist()
todo_c=[r for r in repos if r not in cdone]
print(f"Commit panels to collect: {len(todo_c)}")
for i,fn in enumerate(todo_c):
    try:
        mc=monthly_commit_counts(fn)
        for ym,n in mc.items():
            append_jsonl(commits_panel_path, {"full_name":fn,"ym":ym,"commits":n})
    except Exception as e:
        append_jsonl(commits_panel_path, {"full_name":fn,"error":str(e)})
    cdone.add(fn); led["commits_done"]=sorted(cdone)
    if i%25==0: save_ledger(act_led,led); print(f"  commits [{i+1}/{len(todo_c)}] {fn}")
save_ledger(act_led,led)

todo_h=[r for r in repos if r not in hdone]
print(f"Churn panels to collect: {len(todo_h)}")
for i,fn in enumerate(todo_h):
    cf=weekly_code_frequency(fn)
    for row in cf:
        try:
            wk=pd.to_datetime(row[0],unit='s',utc=True)
            if W0<=wk<=W1:
                append_jsonl(churn_panel_path,
                    {"full_name":fn,"week":wk.isoformat(),
                     "additions":row[1],"deletions":abs(row[2])})
        except Exception: pass
    hdone.add(fn); led["churn_done"]=sorted(hdone)
    if i%25==0: save_ledger(act_led,led); print(f"  churn [{i+1}/{len(todo_h)}] {fn}")
save_ledger(act_led,led)
print("Activity collection complete.")


---
## Cell 4 — Collect PR volume → monthly panel (all cohort repos)

Monthly counts of pull requests **created**, via the issues/PR search or the list-PRs endpoint paged by `created` date. PR history does not age out → full window.


In [ ]:

cohort = pd.read_parquet(DIRS['cohort']/'cohort.parquet')
pr_panel_path = DIRS['panels']/'prs_monthly.jsonl'
pr_led = DIRS['panels']/'pr_ledger.json'
led = load_ledger(pr_led, {"done": []}); pdone=set(led["done"])

W0=CONFIG['study_window'][0]; W1=CONFIG['study_window'][1]

def monthly_pr_counts(full_name):
    counts={}; page=1
    while page<=20:
        prs=gh_json(f"{API}/repos/{full_name}/pulls",
                    params={"state":"all","per_page":100,"page":page,
                            "sort":"created","direction":"asc"})
        if not prs: break
        for pr in prs:
            d=dtp.parse(pr["created_at"])
            if W0<=d.strftime("%Y-%m-%d")<=W1:
                key=f"{d.year}-{d.month:02d}"; counts[key]=counts.get(key,0)+1
        if len(prs)<100: break
        page+=1
    return counts

repos=cohort['full_name'].tolist()
todo=[r for r in repos if r not in pdone]
print(f"PR panels to collect: {len(todo)}")
for i,fn in enumerate(todo):
    try:
        mc=monthly_pr_counts(fn)
        for ym,n in mc.items():
            append_jsonl(pr_panel_path,{"full_name":fn,"ym":ym,"prs":n})
    except Exception as e:
        append_jsonl(pr_panel_path,{"full_name":fn,"error":str(e)})
    pdone.add(fn); led["done"]=sorted(pdone)
    if i%25==0: save_ledger(pr_led,led); print(f"  prs [{i+1}/{len(todo)}] {fn}")
save_ledger(pr_led,led)
print("PR collection complete.")


---
## Cell 5 — Collect CI run durations → monthly CI-minutes panel (CI-eligible repos)

For repos with ≥ 9 months retrievable CI history, we page the Actions runs API and sum **run durations** (updated_at − run_started_at) into **monthly CI-minutes**. This is the primary-outcome backbone for RQ2; it carries the retention-window boundary. Resumable per repo.

> Duration is wall-clock per run, the input proxy for compute. The minutes→energy→CO2e conversion (Green Algorithms / SCI / CCF, with a low/central/high sensitivity band) happens in the carbon notebook, not here.


In [ ]:

cohort = pd.read_parquet(DIRS['cohort']/'cohort.parquet')
ci_repos = cohort[cohort['ci_eligible']]['full_name'].tolist()
ci_panel_path = DIRS['panels']/'ci_minutes_monthly.jsonl'
ci_led = DIRS['panels']/'ci_ledger.json'
led = load_ledger(ci_led, {"done": []}); cidone=set(led["done"])

def monthly_ci_minutes(full_name):
    agg={}; page=1
    while page<=50:   # up to 5000 runs
        data=gh_json(f"{API}/repos/{full_name}/actions/runs",
                     params={"per_page":100,"page":page})
        runs=data.get("workflow_runs",[])
        if not runs: break
        for r in runs:
            st=r.get("run_started_at"); en=r.get("updated_at")
            if not st or not en: continue
            try:
                t0=dtp.parse(st); t1=dtp.parse(en)
                mins=max((t1-t0).total_seconds()/60.0, 0.0)
            except Exception: continue
            key=f"{t0.year}-{t0.month:02d}"
            a=agg.setdefault(key,{"ci_minutes":0.0,"ci_runs":0})
            a["ci_minutes"]+=mins; a["ci_runs"]+=1
        if len(runs)<100: break
        page+=1
    return agg

todo=[r for r in ci_repos if r not in cidone]
print(f"CI-minutes panels to collect (eligible repos): {len(todo)}")
for i,fn in enumerate(todo):
    try:
        agg=monthly_ci_minutes(fn)
        for ym,v in agg.items():
            append_jsonl(ci_panel_path,
                {"full_name":fn,"ym":ym,
                 "ci_minutes":round(v["ci_minutes"],2),"ci_runs":v["ci_runs"]})
    except Exception as e:
        append_jsonl(ci_panel_path,{"full_name":fn,"error":str(e)})
    cidone.add(fn); led["done"]=sorted(cidone)
    if i%20==0: save_ledger(ci_led,led); print(f"  ci [{i+1}/{len(todo)}] {fn}")
save_ledger(ci_led,led)
print("CI-minutes collection complete.")


---
## Cell 6 — Assemble tidy `repo × month` panels with event time

We merge the collected pieces into analysis-ready long panels keyed by `(full_name, ym)`, attach `is_treated`, the (pseudo-)adoption date, **event-time months** (months relative to adoption), and a `post` indicator. Missing months are filled as zero activity within each repo's observed span. These panels are what Notebook 3B's DiD consumes.


In [ ]:

cohort = pd.read_parquet(DIRS['cohort']/'cohort.parquet')[
    ['full_name','language','band','is_treated','adoption_date','adoption_source',
     'ci_eligible','stars']]
cohort['adoption_date'] = pd.to_datetime(cohort['adoption_date'], utc=True)

def load_panel(path, value_cols):
    df=pd.DataFrame(load_jsonl(path))
    if 'error' in df.columns: df=df[df['error'].isna()] if df['error'].notna().any() else df
    return df.dropna(subset=['ym']) if 'ym' in df.columns else df

commits=load_panel(DIRS['panels']/'commits_monthly.jsonl',['commits'])
prs=load_panel(DIRS['panels']/'prs_monthly.jsonl',['prs'])
ci=load_panel(DIRS['panels']/'ci_minutes_monthly.jsonl',['ci_minutes','ci_runs'])

# churn weekly -> monthly
chw=pd.DataFrame(load_jsonl(DIRS['panels']/'churn_weekly.jsonl'))
if len(chw) and 'week' in chw.columns:
    chw['week']=pd.to_datetime(chw['week'],utc=True)
    chw['ym']=chw['week'].dt.strftime('%Y-%m')
    churn=chw.groupby(['full_name','ym'],as_index=False).agg(
        additions=('additions','sum'),deletions=('deletions','sum'))
    churn['churn']=churn['additions']+churn['deletions']
else:
    churn=pd.DataFrame(columns=['full_name','ym','churn'])

panel=commits.merge(prs,on=['full_name','ym'],how='outer') \
             .merge(churn[['full_name','ym','churn']],on=['full_name','ym'],how='outer') \
             .merge(ci[['full_name','ym','ci_minutes','ci_runs']] if len(ci) else
                    pd.DataFrame(columns=['full_name','ym','ci_minutes','ci_runs']),
                    on=['full_name','ym'],how='outer')
for c in ['commits','prs','churn','ci_minutes','ci_runs']:
    if c in panel: panel[c]=panel[c].fillna(0)

panel=panel.merge(cohort,on='full_name',how='left')
panel['month_ts']=pd.to_datetime(panel['ym']+'-01',utc=True)
panel['event_month']=((panel['month_ts'].dt.year-panel['adoption_date'].dt.year)*12
                      +(panel['month_ts'].dt.month-panel['adoption_date'].dt.month))
panel['post']=(panel['event_month']>=0).astype(int)

PRE=CONFIG['pre_period_days']//30; POST=CONFIG['post_period_days']//30
inwin=panel['event_month'].between(-PRE,POST-1)
panel_win=panel[inwin].copy()

panel.to_parquet(DIRS['panels']/'panel_full.parquet',index=False)
panel_win.to_parquet(DIRS['panels']/'panel_windowed.parquet',index=False)

print("PANELS BUILT")
print(f"  full panel rows:      {len(panel)}")
print(f"  windowed panel rows:  {len(panel_win)}  (event_month in [-{PRE},{POST-1}])")
print(f"  repos in panel:       {panel['full_name'].nunique()}")
print(f"  treated:              {panel[panel['is_treated']]['full_name'].nunique()}")
print(f"  with CI minutes >0:   {panel[panel['ci_minutes']>0]['full_name'].nunique()}")
print("\nOutcome coverage (non-zero repo-months):")
for c in ['commits','prs','churn','ci_minutes','ci_runs']:
    if c in panel: print(f"  {c:12s}: {int((panel[c]>0).sum())} repo-months")


---
## Cell 7 — Manifest & handoff to Notebook 3B


In [ ]:

manifest={
  "notebook":"03A_adoption_dating_and_panels",
  "produced":{
    "cohort/adoption_dates.parquet": (DIRS['cohort']/'adoption_dates.parquet').exists(),
    "cohort/cohort.parquet": (DIRS['cohort']/'cohort.parquet').exists(),
    "panels/panel_full.parquet": (DIRS['panels']/'panel_full.parquet').exists(),
    "panels/panel_windowed.parquet": (DIRS['panels']/'panel_windowed.parquet').exists(),
  },
  "timestamp": datetime.datetime.utcnow().isoformat()+"Z",
}
(PROJECT_ROOT/'manifest_nb03A.json').write_text(json.dumps(manifest,indent=2))
print(json.dumps(manifest,indent=2))
print("\nNotebook 3B (next): propensity/NN matching + covariate-balance (SMD) table,")
print("two-way fixed-effects DiD on activity outcomes (all treated) and CI-minutes")
print("(CI-eligible subset), event-study pre-trend plot, BH correction, placebo test.")
